In [1]:
# Install needed libraries
%pip install -U python-jobspy
%pip install tqdm
%pip install xlsxwriter
# Install MongoDB Python driver
%pip install pymongo
%pip install dotenv
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from jobspy import scrape_jobs
from datetime import datetime, date
import numpy as np
from operations import connect_to_mongodb, setup_output_directory

In [3]:
connect_to_mongodb()

Looking for .env at: c:\Users\gianlu\Market-Scraper\Market-Scraper\.env
❌ Error connecting to MongoDB: SSL handshake failed: ac-fky0ob9-shard-00-01.ncfzs7b.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1006) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),SSL handshake failed: ac-fky0ob9-shard-00-02.ncfzs7b.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1006) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),SSL handshake failed: ac-fky0ob9-shard-00-00.ncfzs7b.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1006) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 5.0s, Topology Description: <TopologyDescription id: 692ee9d14eb0d3be96e3f405, topology_type: ReplicaSetNoPrimary, servers: [<ServerDescription ('ac-fky0ob9-shard-00-00.ncfzs7b.mongodb.net', 27017) server_type

In [4]:
# --- 1. Definir Directorio de Salida ---
output_dir = setup_output_directory()
print(f"Directorio de salida: {output_dir}")

Directorio de salida: ..\data\raw\jobs_20251202_102958


In [5]:
# --- 2. Definir Parámetros de Búsqueda Base ---
#roles_buscados = ["Data Scientist", "Machine Learning Engineer", "Data Analyst"]
sectores_clave = ["Fintech", "EdTech", "Future of Work"]
#search_terms = list(product(roles_buscados, sectores_clave))
search_terms = sectores_clave
location = "Remote"

indeed_glassdoor_countries = [
    "Australia",
    "Austria",
    "Belgium",
    "Brazil",
    "Canada",
    "France",
    "Germany",
    "Hong Kong",
    "India",
    "Ireland",
    "Italy",
    "Mexico",
    "Netherlands",
    "New Zealand",
    "Singapore",
    "Spain",
    "Switzerland",
    "UK",
    "USA",
    "Vietnam"
]
country_code = "USA" # Código de país para Indeed/Glassdoor

In [6]:
# --- 3. Lista para guardar resultados ---
# Guardaremos los DataFrames de cada sitio aquí
all_jobs_dfs = []

print("Parámetros listos. Iniciaremos scrapers secuenciales y especializados.")

Parámetros listos. Iniciaremos scrapers secuenciales y especializados.


In [7]:
# --- 1. Scraper: Indeed (El "Caballo de batalla") ---
# Es el más estable y sin límites de solicitudes
print("\n--- Iniciando Scraper: Indeed/Glassdoor ---")
for country_indeed in indeed_glassdoor_countries:
    print(f"Buscando en {country_indeed}")
    for search_term in search_terms:
        #search_term = f'"{item[0]}" AND "{item[1]}"'
        print(f"Buscando: {search_term}")
        try:
            indeed_jobs = scrape_jobs(
                site_name=["indeed","glassdoor"],
                search_term=search_term, # Podemos afinar el término
                location=location,
                country_indeed=country_indeed,
                results_wanted=2000, # Le pedimos más porque es estable
                hours_old=720
            )
            if indeed_jobs is not None and not indeed_jobs.empty:
                print(f"✅ Indeed encontró {len(indeed_jobs)} trabajos.")
                all_jobs_dfs.append(indeed_jobs)
        except Exception as e:
            print(f"❌ Error en Indeed: {e}")


--- Iniciando Scraper: Indeed/Glassdoor ---
Buscando en Australia
Buscando: Fintech
✅ Indeed encontró 8 trabajos.
Buscando: EdTech
✅ Indeed encontró 4 trabajos.
Buscando: Future of Work
✅ Indeed encontró 117 trabajos.
Buscando en Austria
Buscando: Fintech
✅ Indeed encontró 2 trabajos.
Buscando: EdTech
Buscando: Future of Work
✅ Indeed encontró 12 trabajos.
Buscando en Belgium
Buscando: Fintech
Buscando: EdTech
Buscando: Future of Work
✅ Indeed encontró 18 trabajos.
Buscando en Brazil
Buscando: Fintech
✅ Indeed encontró 158 trabajos.
Buscando: EdTech
✅ Indeed encontró 89 trabajos.
Buscando: Future of Work
✅ Indeed encontró 107 trabajos.
Buscando en Canada
Buscando: Fintech
✅ Indeed encontró 122 trabajos.
Buscando: EdTech
✅ Indeed encontró 17 trabajos.
Buscando: Future of Work
✅ Indeed encontró 599 trabajos.
Buscando en France
Buscando: Fintech
✅ Indeed encontró 533 trabajos.
Buscando: EdTech
✅ Indeed encontró 121 trabajos.
Buscando: Future of Work
✅ Indeed encontró 292 trabajos.
Buscan

In [8]:
"""
# --- 3. Scraper: ZipRecruiter (El "Estándar") ---
print("\n--- Iniciando Scraper: ZipRecruiter ---")
for item in search_terms:
    search_term = f'"{item[0]}" AND "{item[1]}"'
    print(f"Buscando: {search_term}")
    try:
        zip_jobs = scrape_jobs(
            site_name=["zip_recruiter"],
            search_term=search_term,
            location=location,
            results_wanted=100, 
            hours_old=720
        )
        if zip_jobs is not None and not zip_jobs.empty:
            print(f"✅ ZipRecruiter encontró {len(zip_jobs)} trabajos.")
            all_jobs_dfs.append(zip_jobs)
    except Exception as e:
        print(f"❌ Error en ZipRecruiter: {e}")
"""


'\n# --- 3. Scraper: ZipRecruiter (El "Estándar") ---\nprint("\n--- Iniciando Scraper: ZipRecruiter ---")\nfor item in search_terms:\n    search_term = f\'"{item[0]}" AND "{item[1]}"\'\n    print(f"Buscando: {search_term}")\n    try:\n        zip_jobs = scrape_jobs(\n            site_name=["zip_recruiter"],\n            search_term=search_term,\n            location=location,\n            results_wanted=100, \n            hours_old=720\n        )\n        if zip_jobs is not None and not zip_jobs.empty:\n            print(f"✅ ZipRecruiter encontró {len(zip_jobs)} trabajos.")\n            all_jobs_dfs.append(zip_jobs)\n    except Exception as e:\n        print(f"❌ Error en ZipRecruiter: {e}")\n'

In [9]:
"""print("\n--- Iniciando Scraper: Glassdoor ---")
for item in search_terms:
    search_term = f'"{item[0]}" AND "{item[1]}"'
    print(f"Buscando: {search_term}")
    try:
        zip_jobs = scrape_jobs(
            site_name=["glassdoor"],
            search_term=search_term,
            location=location,
            results_wanted=100, 
            hours_old=720
        )
        if zip_jobs is not None and not zip_jobs.empty:
            print(f"✅ GlassDoor encontró {len(zip_jobs)} trabajos.")
            all_jobs_dfs.append(zip_jobs)
    except Exception as e:
        print(f"❌ Error en Glassdoor: {e}")"""

'print("\n--- Iniciando Scraper: Glassdoor ---")\nfor item in search_terms:\n    search_term = f\'"{item[0]}" AND "{item[1]}"\'\n    print(f"Buscando: {search_term}")\n    try:\n        zip_jobs = scrape_jobs(\n            site_name=["glassdoor"],\n            search_term=search_term,\n            location=location,\n            results_wanted=100, \n            hours_old=720\n        )\n        if zip_jobs is not None and not zip_jobs.empty:\n            print(f"✅ GlassDoor encontró {len(zip_jobs)} trabajos.")\n            all_jobs_dfs.append(zip_jobs)\n    except Exception as e:\n        print(f"❌ Error en Glassdoor: {e}")'

In [10]:
# --- 2. Scraper: LinkedIn (El "Delicado") ---
# Alto riesgo de 429. Lo llamamos con cuidado.
"""print("\n--- Iniciando Scraper: LinkedIn ---")
for item in search_terms:
    search_term = f'"{item[0]}" AND "{item[1]}"'
    print(f"Buscando: {search_term}")
    try:
        linkedin_jobs = scrape_jobs(
            site_name=["linkedin"],
            search_term=search_term,
            location=location,
            results_wanted=50, # MUY BAJO para evitar 429 sin proxies
            hours_old=720,
            linkedin_fetch_description=True # Clave para enriquecimiento
        )
        if linkedin_jobs is not None and not linkedin_jobs.empty:
            print(f"✅ LinkedIn encontró {len(linkedin_jobs)} trabajos.")
            all_jobs_dfs.append(linkedin_jobs)
    except Exception as e:
        print(f"❌ Error en LinkedIn: {e}. (Probablemente Error 429)")
"""

'print("\n--- Iniciando Scraper: LinkedIn ---")\nfor item in search_terms:\n    search_term = f\'"{item[0]}" AND "{item[1]}"\'\n    print(f"Buscando: {search_term}")\n    try:\n        linkedin_jobs = scrape_jobs(\n            site_name=["linkedin"],\n            search_term=search_term,\n            location=location,\n            results_wanted=50, # MUY BAJO para evitar 429 sin proxies\n            hours_old=720,\n            linkedin_fetch_description=True # Clave para enriquecimiento\n        )\n        if linkedin_jobs is not None and not linkedin_jobs.empty:\n            print(f"✅ LinkedIn encontró {len(linkedin_jobs)} trabajos.")\n            all_jobs_dfs.append(linkedin_jobs)\n    except Exception as e:\n        print(f"❌ Error en LinkedIn: {e}. (Probablemente Error 429)")\n'

In [11]:
print("\n--- Scraping secuencial completado ---")


--- Scraping secuencial completado ---


In [12]:
def convert_dates_to_datetime(obj):
    """Recursively convert date objects to datetime objects for MongoDB compatibility."""
    if isinstance(obj, dict):
        return {k: convert_dates_to_datetime(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [convert_dates_to_datetime(item) for item in obj]
    elif isinstance(obj, date) and not isinstance(obj, datetime):
        return datetime.combine(obj, datetime.min.time())
    return obj

In [13]:
def process_job_data(job):
    """Process a single job dictionary for MongoDB insertion."""
    # Convert dates and handle NaN/None values
    job = convert_dates_to_datetime(job)
    
    # Convert numpy types to native Python types
    for key, value in job.items():
        if pd.isna(value) or value is None:
            job[key] = None
        elif isinstance(value, (np.generic, np.ndarray)):
            job[key] = value.item() if value.size == 1 else value.tolist()
    
    return job


In [14]:
# Update your main processing loop:
if all_jobs_dfs:
    combined_jobs = pd.concat(all_jobs_dfs, ignore_index=True).drop_duplicates(
        subset=['job_url', 'title', 'company']
    )
    
    # Convert DataFrame to list of dictionaries
    jobs_data = combined_jobs.to_dict('records')
    
    # Connect to MongoDB
    db_connection = connect_to_mongodb()
    
    if db_connection is not None:
        client = db_connection['client']
        collection = db_connection['collection']
        
        try:
            inserted_count = 0
            updated_count = 0
            skipped_count = 0
            errors = []
            
            for job in jobs_data:
                try:
                    # Process the job data
                    processed_job = process_job_data(job)
                    
                    result = collection.update_one(
                        {"job_url": processed_job["job_url"]},
                        {
                            "$set": processed_job,
                            "$setOnInsert": {"created_at": datetime.now()}
                        },
                        upsert=True
                    )
                    
                    if result.upserted_id:
                        inserted_count += 1
                    elif result.modified_count > 0:
                        updated_count += 1
                    else:
                        skipped_count += 1
                        
                except Exception as e:
                    errors.append({
                        "job_url": job.get("job_url", "unknown"),
                        "error": str(e)
                    })
                    print(f"⚠️ Error processing job {job.get('job_url')}: {str(e)}")
                    continue
            
            # Print summary
            print("\n" + "="*50)
            print("📊 Job Processing Summary")
            print("="*50)
            print(f"✅ New jobs inserted: {inserted_count}")
            print(f"🔄 Existing jobs updated: {updated_count}")
            print(f"⏩ Jobs unchanged (skipped): {skipped_count}")
            print(f"❌ Errors: {len(errors)}")
            
            if errors:
                print("\n⚠️  Errors encountered:")
                for i, error in enumerate(errors[:5], 1):  # Show first 5 errors
                    print(f"{i}. {error['job_url']}: {error['error']}")
                if len(errors) > 5:
                    print(f"... and {len(errors) - 5} more errors")
                
                # Save errors to a file
                error_file = f"{output_dir}/import_errors_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
                with open(error_file, 'w') as f:
                    json.dump(errors, f, indent=2)
                print(f"\n📝 Full error log saved to: {error_file}")
            
        except Exception as e:
            print(f"❌ Fatal error: {str(e)}")
            raise
        finally:
            # Always close the connection when done
            client.close()
    else:
        print("❌ Could not connect to MongoDB. Saving to JSON instead...")
    
    # Save backup to JSON
    json_filename = f"{output_dir}/backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    combined_jobs.to_json(json_filename, orient='records', indent=4, date_format='iso')
    print(f"📦 Backup saved to: {json_filename}")
else:
    print("\nNo jobs were found.")

Looking for .env at: c:\Users\gianlu\Market-Scraper\Market-Scraper\.env


C:\Users\gianlu\AppData\Local\Temp\ipykernel_4760\1692231382.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_jobs = pd.concat(all_jobs_dfs, ignore_index=True).drop_duplicates(


❌ Error connecting to MongoDB: SSL handshake failed: ac-fky0ob9-shard-00-01.ncfzs7b.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1006) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),SSL handshake failed: ac-fky0ob9-shard-00-02.ncfzs7b.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1006) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),SSL handshake failed: ac-fky0ob9-shard-00-00.ncfzs7b.mongodb.net:27017: [SSL: TLSV1_ALERT_INTERNAL_ERROR] tlsv1 alert internal error (_ssl.c:1006) (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 5.0s, Topology Description: <TopologyDescription id: 692eebe64eb0d3be96e3f406, topology_type: ReplicaSetNoPrimary, servers: [<ServerDescription ('ac-fky0ob9-shard-00-00.ncfzs7b.mongodb.net', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('SSL handshake failed: ac-fky0